# Arbre de regression - Prevision horaire

Le modele utilise le decoupage chronologique 80/20, les variables originales et les variables retardees `lag_1`, `lag_24` et `lag_168`.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
print(BASE_DIR)

c:\Users\dell\Desktop\arbre de regression


## 1. Preparation et modelisation

L'heure manquante est geree par une serie horaire complete. Les lags sont calcules avant le split par decalage temporel; ils ne reutilisent donc pas la valeur cible courante.

In [2]:
%matplotlib inline
%run arbre_regression.py

Lignes totales: 16897
Train: 13517 | Test: 3380
Periode train: 2011-01-08 -> 2012-08-08
Periode test: 2012-08-08 -> 2012-12-31
Variables d'entree: ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'lag_1', 'lag_24', 'lag_168', 'lag_mean_24_168', 'lag_delta_week']

Arbre de base - apprentissage:
{
  "MAE": 0.0,
  "RMSE": 0.0,
  "R2": 1.0
}
Profondeur du modele de base: 32
Feuilles du modele de base: 12432

Meilleure profondeur: 10
Meilleur min_samples_leaf: 3
Meilleur min_samples_split: 10
Meilleur critere: squared_error
RMSE CV moyenne: 64.6436

Arbre optimise - apprentissage:
{
  "MAE": 22.313489324834453,
  "RMSE": 35.885081906719535,
  "R2": 0.9542353811790774
}

Arbre optimise - test:
{
  "MAE": 40.31088204644742,
  "RMSE": 66.55384049473368,
  "R2": 0.9091061219294538
}

Erreurs par tranche:
       tranche       MAE       RMSE  Nombre_observations
     Nuit 0-5h  9.834815  21.767712                  828
    Matin 

## 2. Resultats final

In [3]:
import json

with open(BASE_DIR / "resultats_arbre.json", "r", encoding="utf-8") as handle:
    resultats = json.load(handle)

pd.DataFrame([resultats["test_metrics"] | {
    "max_depth": resultats["max_depth"],
    "min_samples_leaf": resultats["min_samples_leaf"],
    "min_samples_split": resultats["min_samples_split"],
    "criterion": resultats["criterion"]
}])

,MAE,RMSE,R2,max_depth,min_samples_leaf,min_samples_split,criterion
0,40.310882,66.55384,0.909106,10,3,10,squared_error


## 3. Validation croisee

In [4]:
pd.read_csv(BASE_DIR / "cv_max_depth.csv")

,max_depth,min_samples_leaf,min_samples_split,criterion,RMSE_CV,RMSE_CV_std
0,6,2,2,squared_error,70.341559,13.190522
1,6,2,10,squared_error,69.676005,12.625729
2,6,2,20,squared_error,70.398187,13.197680
3,6,3,2,squared_error,70.191787,12.893246
4,6,3,10,squared_error,69.762735,12.748172
...,...,...,...,...,...,...
103,20,10,10,squared_error,67.096931,13.109912
104,20,10,20,squared_error,67.096931,13.109912
105,20,15,2,squared_error,66.786528,12.700653
106,20,15,10,squared_error,66.786528,12.700653


## 4. Erreurs par heure et par tranche

In [5]:
display(pd.read_csv(BASE_DIR / "erreurs_par_heure.csv"))

display(pd.read_csv(BASE_DIR / "erreurs_par_tranche.csv"))

,hr,MAE,RMSE,Nombre_observations
0,0,21.278159,38.500293,142
1,1,11.516786,29.542814,140
2,2,7.337895,12.990803,140
3,3,5.897557,9.688672,132
4,4,3.396322,4.583702,135
5,5,8.957494,12.235264,139
6,6,22.451123,34.346940,140
7,7,39.821633,64.236471,140
8,8,61.464837,82.160757,140
9,9,51.170669,71.886371,140


,tranche,MAE,RMSE,Nombre_observations
0,Nuit 0-5h,9.834815,21.767712,828
1,Matin 6-9h,43.727066,65.620265,560
2,Journee 10-15h,54.147715,81.680430,848
3,Soir 16-19h,73.158952,100.108851,572
4,Soiree 20-23h,27.720656,36.871986,572


## 5. Importance des variables

In [6]:
pd.read_csv(BASE_DIR / "importance_variables.csv")

,Variable,Importance
0,lag_mean_24_168,0.697657
1,lag_1,0.211034
2,hr,0.035053
3,lag_168,0.023093
4,workingday,0.010292
5,lag_delta_week,0.005215
6,weathersit,0.004872
7,lag_24,0.003722
8,hum,0.002157
9,temp,0.001479


## 6. Interpretation

- La variable retardee moyenne `lag_mean_24_168` est la plus importante, suivie de `lag_1`, ce qui confirme l'apport des informations historiques.
- La meilleure configuration est `max_depth=10`, `min_samples_leaf=3`, `min_samples_split=10` et `criterion=squared_error`.
- RMSE de validation croisee: 64.6436.
- Sur l'ensemble d'apprentissage: MAE = 22.3135, RMSE = 35.8851, R2 = 0.954235.
- Sur le test chronologique: MAE = 40.3109, RMSE = 66.5538, R2 = 0.909106.
- L'objectif minimum de R2 = 0.90 est atteint sur le test.
- La plus forte erreur reste dans la tranche de 16h a 19h, mais elle est nettement reduite par rapport au modele sans variables retardees.
- L'arbre illimite garde un score d'apprentissage parfait, mais ne constitue qu'un point de reference pour illustrer le surapprentissage.